# 02_governance
Captured 2026-11-21 21:06 UTC

**Same query, two identities.** The supply executive sees all regions and revenue; the `planner_eu` service principal sees only EU rows and NULL revenue (row filter `governance.region_filter`, mask `governance.mask_commercial`). Silver and bronze are not granted to planners.


In [1]:
-- as supply_exec (all regions, commercial visible)
SELECT region, count(*) AS rows, count(annual_revenue) AS revenue_visible FROM supply_chain.gold.days_of_cover GROUP BY region ORDER BY region

region | rows | revenue_visible
----------------------------------
APAC   | 98   | 98             
EU     | 90   | 90             
US     | 100  | 100            
(3 rows)

In [2]:
-- as planner_eu service principal
SELECT region, count(*) AS rows, count(annual_revenue) AS revenue_visible FROM supply_chain.gold.days_of_cover GROUP BY region ORDER BY region

region | rows | revenue_visible
----------------------------------
EU     | 90   | 0              
(1 rows)

In [3]:
-- disruption impact as supply_exec
SELECT sku_id, region, supply_gap_days, revenue_at_risk FROM supply_chain.gold.disruption_impact ORDER BY supply_gap_days DESC LIMIT 5

sku_id   | region | supply_gap_days | revenue_at_risk
--------------------------------------------------------
SKU-0118 | EU     | 127.7           | 5582913.0      
SKU-0119 | US     | 117.9           | 1938810.0      
SKU-0083 | APAC   | 115.4           | 3906392.0      
SKU-0122 | EU     | 96.4            | 2821370.0      
SKU-0088 | US     | 90.5            | 410684.0       
(5 rows)

In [4]:
-- disruption impact as planner_eu
SELECT sku_id, region, supply_gap_days, revenue_at_risk FROM supply_chain.gold.disruption_impact ORDER BY supply_gap_days DESC LIMIT 5

sku_id   | region | supply_gap_days | revenue_at_risk
--------------------------------------------------------
SKU-0118 | EU     | 127.7           | NULL           
SKU-0122 | EU     | 96.4            | NULL           
SKU-0113 | EU     | 49.1            | NULL           
SKU-0116 | EU     | 30.3            | NULL           
SKU-0127 | EU     | 0.0             | NULL           
(5 rows)

In [5]:
-- planner cannot read the unfiltered calc layer
SELECT count(*) FROM supply_chain.silver.impact_base

FAILED: [INSUFFICIENT_PERMISSIONS] Insufficient privileges:
Principal 'shortage-radar-planner-eu' does not have USE SCHEMA on Schema 'supply_chain.silver'. SQLSTATE: 42501

In [6]:
-- entitlement table (deny by default)
SELECT persona, region, can_see_commercial, can_approve FROM supply_chain.governance.user_entitlement

persona     | region | can_see_commercial | can_approve
----------------------------------------------------------
planner_eu  | EU     | false              | false      
supply_exec | *      | true               | true       
(2 rows)